# 02 — Data Preparation & User-Item Matrix Building

This notebook covers the full data preparation pipeline:
1. Loading and validating the synthetic dataset
2. Feature engineering (credit score derivation, categorical encoding)
3. Building the user-item interaction matrix
4. Train/test split for held-out evaluation
5. Saving processed artefacts for downstream modelling


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

print('Config loaded:', config['project']['name'])
print('Data paths:', config.get('data', {}))

In [ ]:
# ── 1. Load synthetic data ──────────────────────────────────────────────────
from src.data.synthetic_generator import SyntheticSMEGenerator

gen = SyntheticSMEGenerator(n_smes=5000, random_state=42)
data = gen.generate()

sme_features   = data['sme_features'].copy()
user_item_matrix = data['user_item_matrix'].copy()
ratings_long   = data['ratings_long'].copy()
item_features  = data['item_features'].copy()

print('Shapes:')
print(f'  sme_features:     {sme_features.shape}')
print(f'  user_item_matrix: {user_item_matrix.shape}')
print(f'  ratings_long:     {ratings_long.shape}')
print(f'  item_features:    {item_features.shape}')

In [ ]:
# ── 2. Data validation ───────────────────────────────────────────────────────
print('=== SME Features ===')
print(sme_features.dtypes)
print('\nMissing values:')
print(sme_features.isnull().sum())

print('\n=== Ratings Long ===')
print(ratings_long.dtypes)
print('Rating range:', ratings_long['rating'].min(), '-', ratings_long['rating'].max())
print('Unique products:', sorted(ratings_long['product_id'].unique()))
print('Unique SMEs with interactions:', ratings_long['sme_id'].nunique())

print('\n=== Item Features ===')
print(item_features)

In [ ]:
# ── 3. Feature engineering ───────────────────────────────────────────────────
# 3a. Derived credit score (if not already present)
if 'credit_score' not in sme_features.columns:
    rev_norm  = np.clip((sme_features['annual_revenue_usd'] - 500) / 49500, 0, 1)
    age_norm  = np.clip(sme_features['years_in_business'] / 20, 0, 1)
    loan_flag = sme_features['previous_loan'].astype(float)
    bank_flag = sme_features['has_bank_account'].astype(float)
    sme_features['credit_score'] = (
        0.40 * rev_norm + 0.25 * age_norm + 0.20 * loan_flag + 0.15 * bank_flag
    ) * 850  # scale to 0-850
    sme_features['credit_score'] = sme_features['credit_score'].round(0).astype(int)
    print('Credit score derived.')
else:
    print('Credit score already present.')

print(sme_features[['sme_id', 'credit_score', 'annual_revenue_usd']].describe())

In [ ]:
# 3b. Categorical encoding
from sklearn.preprocessing import LabelEncoder

cat_cols = ['country', 'sector', 'gender']
label_encoders = {}

sme_encoded = sme_features.copy()
for col in cat_cols:
    if col in sme_encoded.columns:
        le = LabelEncoder()
        sme_encoded[col + '_enc'] = le.fit_transform(sme_encoded[col].astype(str))
        label_encoders[col] = le
        print(f'  {col}: {list(le.classes_)}')

# One-hot encoding for modelling
sme_ohe = pd.get_dummies(sme_features, columns=cat_cols, drop_first=False)
print(f'\nOne-hot encoded shape: {sme_ohe.shape}')
sme_ohe.head(2)

In [ ]:
# 3c. Normalise numeric features
from sklearn.preprocessing import MinMaxScaler

numeric_cols = ['annual_revenue_usd', 'credit_score', 'years_in_business',
                'num_employees', 'loan_amount_requested']
numeric_cols = [c for c in numeric_cols if c in sme_features.columns]

scaler = MinMaxScaler()
sme_features_scaled = sme_features.copy()
sme_features_scaled[numeric_cols] = scaler.fit_transform(sme_features[numeric_cols])

print('Normalised numeric columns:')
sme_features_scaled[numeric_cols].describe().round(3)

In [ ]:
# ── 4. Verify user-item matrix ───────────────────────────────────────────────
print('User-item matrix preview:')
print(user_item_matrix.iloc[:5, :])
print(f'\nIndex type: {type(user_item_matrix.index[0])}')
print(f'Column names (product IDs): {user_item_matrix.columns.tolist()}')

total_cells  = user_item_matrix.size
filled_cells = (user_item_matrix > 0).sum().sum()
sparsity     = 1 - filled_cells / total_cells
print(f'\nSparsity: {sparsity:.2%}')

# Distribution of interactions per SME
interactions_per_sme = (user_item_matrix > 0).sum(axis=1)
print(f'\nInteractions per SME:')
print(interactions_per_sme.value_counts().sort_index())

In [ ]:
# ── 5. Train / test split ────────────────────────────────────────────────────
# Strategy: leave-one-out per user (hold out the most recent interaction)
from sklearn.model_selection import train_test_split

# For simplicity, do a random 80/20 split on ratings_long
np.random.seed(42)
train_ratings, test_ratings = train_test_split(
    ratings_long, test_size=0.20, random_state=42, stratify=ratings_long['product_id']
)

print(f'Train ratings: {train_ratings.shape}')
print(f'Test ratings:  {test_ratings.shape}')

# Build train user-item matrix
train_pivot = train_ratings.pivot_table(
    index='sme_id', columns='product_id', values='rating', aggfunc='mean'
).fillna(0)

# Align columns with full matrix
for col in user_item_matrix.columns:
    if col not in train_pivot.columns:
        train_pivot[col] = 0
train_pivot = train_pivot[user_item_matrix.columns]

print(f'\nTrain pivot shape: {train_pivot.shape}')
print(f'Test SMEs:  {test_ratings["sme_id"].nunique()}')

In [ ]:
# Visualise train vs test product coverage
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

train_ratings['product_id'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white', title='Train: interactions per product'
)
test_ratings['product_id'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white', title='Test: interactions per product'
)
for ax in axes:
    ax.set_xlabel('Product ID')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Save processed data ──────────────────────────────────────────────────
import os, pickle

out_dir = '../data/processed'
os.makedirs(out_dir, exist_ok=True)

# Save as CSV
sme_features.to_csv(f'{out_dir}/sme_features.csv', index=False)
sme_features_scaled.to_csv(f'{out_dir}/sme_features_scaled.csv', index=False)
user_item_matrix.to_csv(f'{out_dir}/user_item_matrix.csv')
train_ratings.to_csv(f'{out_dir}/train_ratings.csv', index=False)
test_ratings.to_csv(f'{out_dir}/test_ratings.csv', index=False)
train_pivot.to_csv(f'{out_dir}/train_pivot.csv')
item_features.to_csv(f'{out_dir}/item_features.csv', index=False)

# Save encoders and scaler
with open(f'{out_dir}/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
with open(f'{out_dir}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved artefacts:')
for fn in os.listdir(out_dir):
    fp = os.path.join(out_dir, fn)
    print(f'  {fn:40s}  {os.path.getsize(fp)/1024:.1f} KB')

## Summary

| Artefact | Description |
|---|---|
| `sme_features.csv` | Raw SME demographic and financial features |
| `sme_features_scaled.csv` | Min-max normalised numeric features |
| `user_item_matrix.csv` | Full 5000 x 8 interaction matrix |
| `train_ratings.csv` | 80% of interactions for training |
| `test_ratings.csv` | 20% held-out interactions for evaluation |
| `train_pivot.csv` | Pivot of training interactions |
| `label_encoders.pkl` | Sklearn LabelEncoders for country/sector/gender |
| `scaler.pkl` | MinMaxScaler fitted on numeric features |

These artefacts are consumed by notebooks 03–07.
